# 06 · Event contracts and schema evolution
**Level:** intermediate · **Time:** 45 minutes · **Prerequisite:** 03–05.
Define a JSON Schema, reject invalid messages, and test a compatible field addition.
This lab validates locally. It does **not** deploy a Schema Registry or use its wire format.
Kafka itself does not enforce this schema. The Django event contract is introduced in lab 11.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "labtools").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the kafka-django workshop folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from labtools.kafka import (
    bootstrap_servers, client_config, require_broker, new_topic,
    producer, consumer, publish, poll_records, collect, decode,
)
RUN = uuid4().hex[:8]
print("Workshop:", ROOT.name, "| run:", RUN)


## 1. Write the contract before the producer
Use a stable event ID, explicit type/version, UTC timestamp, and exact money representation.
Allowing unknown fields gives old readers room to accept additive changes.


In [ ]:
from jsonschema import Draft202012Validator, FormatChecker
schema_v1 = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "required": ["event_id", "event_type", "schema_version", "occurred_at", "order_id", "total"],
    "properties": {
        "event_id": {"type": "string", "format": "uuid"},
        "event_type": {"const": "order.created"},
        "schema_version": {"const": 1},
        "occurred_at": {"type": "string", "format": "date-time"},
        "order_id": {"type": "string", "minLength": 1},
        "total": {"type": "string", "pattern": r"^[0-9]+\.[0-9]{2}$"},
    },
    "additionalProperties": True,
}
Draft202012Validator.check_schema(schema_v1)
validator_v1 = Draft202012Validator(schema_v1, format_checker=FormatChecker())


In [ ]:
from datetime import datetime, timezone
good = {"event_id": str(uuid4()), "event_type": "order.created", "schema_version": 1,
        "occurred_at": datetime.now(timezone.utc).isoformat(), "order_id": "order-contract", "total": "25.00"}
validator_v1.validate(good)
print("Valid event:", good)


## 2. Fail early with useful errors
An integer or float total is not the agreed wire representation. Schema validation is
separate from business validation such as an upper limit or a permitted currency.


In [ ]:
bad = {**good, "event_id": "not-a-uuid", "total": 25.0}
errors = sorted(validator_v1.iter_errors(bad), key=lambda error: str(error.path))
for error in errors:
    print(list(error.path), error.message)
assert len(errors) == 2


## 3. Add an optional field
Here the optional currency has a documented default of USD. JSON Schema's `default`
is an annotation: it does not mutate the event. Consumers must supply the default.
This compatible addition stays on schema version 1; a breaking major version needs a migration.


In [ ]:
from copy import deepcopy
schema_v1_1 = deepcopy(schema_v1)
schema_v1_1["properties"]["currency"] = {"type": "string", "enum": ["USD", "INR", "EUR"], "default": "USD"}
validator_new = Draft202012Validator(schema_v1_1, format_checker=FormatChecker())
new_event = {**good, "event_id": str(uuid4()), "currency": "INR"}
# Old reader accepts a new writer; new reader accepts an old writer.
validator_v1.validate(new_event)
validator_new.validate(good)
assert "currency" not in good
assert good.get("currency", "USD") == "USD"
print("Both sample compatibility directions pass.")


## 4. Show a breaking change
Renaming `total` to `amount` removes a required field. Testing two examples is useful
but not a formal proof of compatibility for all values. Registry compatibility rules
depend on the schema format, policy and reader/writer semantics.


In [ ]:
breaking = {key: value for key, value in good.items() if key != "total"}
breaking["amount"] = "25.00"
assert not validator_v1.is_valid(breaking)
print([error.message for error in validator_v1.iter_errors(breaking)])


## 5. Validate both sides of a real Kafka exchange
Validate producer input, then validate the bytes decoded by the consumer. A producer
outside your codebase might send an incompatible event, so consumer validation matters too.


In [ ]:
require_broker()
topic = new_topic("contracts", partitions=1)
for event in [good, new_event]:
    validator_new.validate(event)
publish(topic, [good, new_event], key_field="order_id")
received = [decode(m) for m in collect(topic, count=2)]
for event in received:
    validator_new.validate(event)
    print(event["order_id"], event["total"], event.get("currency", "USD"))


## Exercise
Make `currency` required and rerun the old-writer validation. Then design a migration
from `total` to `amount`: consider dual fields, a new event type/topic, and rollout order.
<details><summary>Solution hints</summary>
Requiring currency breaks old events lacking it. During migration, old producers may
still run and retained events may be replayed. Update readers first, test historical
fixtures, then update writers. Remove legacy fields only after a deliberate compatibility window.
</details>

## Extension: Schema Registry
A registry stores schemas and applies configured compatibility checks. Its serializers
typically include a schema identifier in a defined wire format; bare JSON in this lesson
is not automatically compatible with those serializers. Add a registry as a separate exercise.
See [Confluent's serialization APIs](https://docs.confluent.io/platform/current/clients/confluent-kafka-python/html/index.html).
Next: **07 — Kafka transactions**, or **08 — retries and dead-letter topics**.
